# Topic Labeling with Qwen using GRPO and LoRA: Fine-tuning for Semantic Generation

---

## Introduction

This notebook demonstrates how to fine-tune the Qwen2.5-3B-Instruct model for a topic labeling task using GRPO (Gradient Reward Policy Optimization). Given a set of keywords describing a topic and a research field, the model learns to generate an appropriate topic label. Unlike traditional classification approaches, this method uses embedding-based cosine similarity as the reward signal, allowing the model to learn semantic equivalence rather than exact string matching. We utilize LoRA for parameter-efficient fine-tuning, vLLM for accelerated inference, and custom reward functions based on sentence embeddings. This approach is particularly useful for generating human-readable topic descriptions that align with domain-specific contexts.

## Step 1: Installing Dependencies

This step installs the necessary Python packages required for fine-tuning Qwen with GRPO for topic labeling. The packages include:
- **unsloth & vllm:** For fast language model loading and inference.
- **triton:** A specific version needed for compatibility.
- **pynvml:** For managing NVIDIA GPU resources.
- **sentence-transformers:** For generating embeddings to compute cosine similarity between labels.
- **scikit-learn:** For cosine similarity computation.

In [ ]:
%%capture
# Install necessary packages for fast model inference and GPU management.
!pip install unsloth vllm  
!pip install triton==3.1.0  
!pip install -U pynvml
!pip install sentence-transformers scikit-learn pandas

## Step 2: Loading the Model with LoRA Adapters

In this step, we load the Qwen model using the FastLanguageModel API from unsloth. We also apply a patch for GRPO using LoRA (Low-Rank Adaptation) with specific parameters. This setup allows efficient model finetuning and fast inference.

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL

# Patch the FastLanguageModel to integrate GRPO-specific modifications.
PatchFastRL("GRPO", FastLanguageModel)

from unsloth import is_bfloat16_supported
import torch

# Set maximum sequence length and LoRA rank (controls the adaptation complexity).
max_seq_length = 1024  # Increase if you need longer reasoning traces.
lora_rank = 64         # Larger rank can improve performance but may slow down training.

# Load the Qwen model in 4-bit mode for reduced memory usage and enable fast inference with vLLM.
model, tokenizer = FastLanguageModel.from_pretrained( 
    model_name = "Qwen/Qwen2.5-3B-Instruct", 
    max_seq_length = max_seq_length, 
    load_in_4bit = True, # Set to False if using LoRA in 16-bit precision. 
    max_lora_rank = lora_rank, 
    gpu_memory_utilization = 0.5, # Adjust GPU memory usage to avoid out-of-memory errors. 
)

# Wrap the model with PEFT (Parameter-Efficient Fine-Tuning) using LoRA.
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,           # Use a rank greater than 0; common choices include 8, 16, 32, 64, or 128.
    lora_alpha = lora_rank,  # A higher lora_alpha value means that the LoRA layers have a greater influence on the model's output, 
                             # while a lower value reduces this influence
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],                                       # Specify target modules; you can remove QKVO if memory is limited.
    use_gradient_checkpointing = "unsloth",  # Enable gradient checkpointing for long context finetuning.
    random_state = 3407,                     # Set a random seed for reproducibility.
)

## Step 3: Preparing the Dataset

This block loads the taxonomy dataset from a CSV file. The dataset contains three columns:
- **topic_words:** Comma-separated keywords describing a topic
- **topic_label:** A descriptive label for the topic
- **research_field:** The research field the topic belongs to

The data is formatted to create prompts that include the topic keywords and research field, with the target being the topic label.

In [ ]:
import pandas as pd
from datasets import Dataset

# Define a system prompt for the topic labeling task.
SYSTEM_PROMPT = """You are an expert at analyzing research topics and generating concise, descriptive labels.
Given a set of keywords and a research field, generate a clear and relevant topic label that captures the essence of the topic.
Provide only the topic label without any additional explanation."""

# Load the taxonomy dataset
df = pd.read_csv('/kaggle/input/datasets/ahmedamrabolfadl/scientific-topic-labeling-dataset/verified_pairs.csv')
df.dropna()
num_samples = df.shape[0]

# Function to prepare the dataset for training.
def prepare_topic_labeling_dataset(df) -> Dataset:
    """
    Prepare the dataset by creating prompts from topic_words and research_field,
    and using topic_label as the target answer.
    """
    data_list = []
    for idx, row in df.iterrows():
        topic_words = row['topic_words']
        research_field = row['research_field']
        topic_label = row['topic_label']
        
        # Create the prompt
        prompt_text = f"Topic Keywords: {topic_words}\nResearch Field: {research_field}"
        
        data_list.append({
            'prompt': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': prompt_text}
            ],
            'expected_label': topic_label
        })
    
    return Dataset.from_list(data_list)

# Create the dataset to be used for training.
dataset = prepare_topic_labeling_dataset(df)

## Step 4: Defining Reward Functions for GRPO

This section defines reward functions for the topic labeling task. Instead of checking exact string matches or XML formatting, these functions use embedding-based cosine similarity to evaluate how well the generated label semantically matches the expected label.

### Explanation of Reward Functions

1. **embedding_similarity_reward_func**  
   - **Purpose:** Compute cosine similarity between the generated label's embedding and the expected label's embedding.  
   - **How it works:**  
     - Uses a pre-trained sentence transformer model to encode both the generated and expected labels.
     - Computes cosine similarity between the two embeddings.
     - Returns the similarity score (0 to 1) as the reward, scaled up to encourage optimization.
     - This reward is model-agnostic and captures semantic similarity rather than exact string matching.

2. **length_penalty_reward_func**  
   - **Purpose:** Encourage reasonable-length outputs without being too verbose.  
   - **How it works:**  
     - Applies a small penalty for outputs that are much longer than the expected label.
     - Rewards outputs that are roughly similar in length to the expected label.
     - Returns a penalty score that can be combined with similarity scores.

3. **combined_reward_func**  
   - **Purpose:** Combine embedding similarity and length penalty for a comprehensive evaluation.  
   - **How it works:**  
     - Weighs the embedding similarity heavily (primary signal).
     - Applies a smaller penalty for length discrepancies.
     - Returns the combined score as the final reward.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load a pre-trained sentence transformer model for embedding generation
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Reward function based on embedding cosine similarity
def embedding_similarity_reward_func(completions, expected_label, **kwargs) -> list[float]:
    """
    Compute cosine similarity between generated and expected labels using sentence embeddings.
    """
    # Extract generated labels from completions
    generated_labels = [completion[0]['content'].strip() for completion in completions]
    expected_labels = [label for label in expected_label]
    
    rewards = []
    for generated, expected in zip(generated_labels, expected_labels):
        # Generate embeddings for both labels
        generated_embedding = embedding_model.encode(generated, convert_to_tensor=False)
        expected_embedding = embedding_model.encode(expected, convert_to_tensor=False)
        
        # Compute cosine similarity
        similarity = cosine_similarity(
            generated_embedding.reshape(1, -1),
            expected_embedding.reshape(1, -1)
        )[0][0]
        
        # Scale similarity to a reward (already between 0 and 1)
        reward = float(similarity)
        rewards.append(reward)
        
        # Print details for monitoring
        if len(generated_labels) > 0 and generated_labels[0] == generated:
            print(f"Generated: {generated[:60]}...")
            print(f"Expected: {expected}")
            print(f"Similarity Score: {similarity:.4f}")
            print("-" * 40)
    
    return rewards

# Reward function for penalizing excessively long outputs
def length_penalty_reward_func(completions, expected_label, **kwargs) -> list[float]:
    """
    Apply a penalty for outputs that are significantly longer than the expected label.
    """
    generated_labels = [completion[0]['content'].strip() for completion in completions]
    expected_labels = [label for label in expected_label]
    
    rewards = []
    for generated, expected in zip(generated_labels, expected_labels):
        expected_len = len(expected.split())
        generated_len = len(generated.split())
        
        # Allow up to 2x the expected length, then apply penalty
        if generated_len <= expected_len + 2:
            penalty = 0.0
        else:
            penalty = -0.1 * (generated_len - expected_len + 2) / expected_len
        
        rewards.append(max(penalty, -0.5))  # Cap the penalty at -0.5
    
    return rewards

# Combined reward function
def combined_reward_func(completions, expected_label, **kwargs) -> list[float]:
    """
    Combine embedding similarity (70%) and length penalty (30%).
    """
    similarity_rewards = embedding_similarity_reward_func(completions, expected_label, **kwargs)
    length_rewards = length_penalty_reward_func(completions, expected_label, **kwargs)
    
    combined_rewards = [0.8 * sim + 0.2 * length for sim, length in zip(similarity_rewards, length_rewards)]
    return combined_rewards

## Step 5: Configuring and Running the Trainer

This step sets up the GRPO finetuning process with a specific configuration. It defines training hyperparameters, logging, and gradient accumulation parameters. The GRPOTrainer is then instantiated with the model, tokenizer, reward functions, and training dataset, and training is initiated.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Configure GRPO training parameters for topic labeling.
# This configuration sets up the training hyperparameters, optimization settings, and inference acceleration via vLLM.
training_args = GRPOConfig(
    use_vllm = False,                     # Enable vLLM to accelerate inference during training.
    learning_rate = 5e-6,                # Set the learning rate for the optimizer.
    adam_beta1 = 0.9,                    # First beta parameter for the AdamW optimizer.
    adam_beta2 = 0.99,                   # Second beta parameter for the AdamW optimizer.
    weight_decay = 0.1,                  # Weight decay to regularize the model and prevent overfitting.
    warmup_ratio = 0.1,                  # Fraction of steps used for learning rate warmup.
    lr_scheduler_type = "cosine",        # Use cosine annealing for the learning rate scheduler.
    optim = "adamw_8bit",                # Use 8-bit AdamW optimizer for memory efficiency.
    logging_steps = 1,                   # Log training information every step.
    bf16 = is_bfloat16_supported(),      # Use bfloat16 precision if supported by the GPU.
    fp16 = not is_bfloat16_supported(),  # Otherwise, fall back to fp16 precision.
    per_device_train_batch_size = 1,     # Batch size per device during training.
    gradient_accumulation_steps = 1,     # Accumulate gradients over this many steps (increase for smoother training if needed).
    num_generations = 8,                 # Number of generations per prompt (reduce if memory issues occur).
    max_prompt_length = 256,             # Maximum length for the input prompt.
    max_completion_length = 100,         # Maximum length for generated label (typically shorter than reasoning tasks).
    # num_train_epochs = 1,               # Uncomment this line to run training for one epoch.
    max_steps = num_samples,                     # Maximum number of training steps.
    save_steps = 250,                    # Save the model checkpoint every specified number of steps.
    max_grad_norm = 0.1,                 # Maximum gradient norm for gradient clipping.
    report_to = "none",                  # Disable reporting to external services like WandB.
    output_dir = "outputs",              # Directory to save the training outputs and checkpoints.
)

# Instantiate the GRPO trainer with the model, tokenizer, reward functions, and training dataset.
trainer = GRPOTrainer(
    model = model,                       # The language model to be trained.
    processing_class = tokenizer,        # The tokenizer used to preprocess the data.
    reward_funcs = [
        combined_reward_func,            # Primary reward function combining similarity and length penalty.
        embedding_similarity_reward_func, # Direct embedding similarity-based reward.
        length_penalty_reward_func,      # Length penalty reward.
    ],
    args = training_args,                # GRPO training configuration.
    train_dataset = dataset,             # The training dataset containing prompts and expected labels.
)

# Begin training using the GRPO algorithm.
trainer.train()

merged_model = model.merge_and_unload()

merged_model.save_pretrained("/kaggle/working/ver_topic_labeler_merged")
tokenizer.save_pretrained("/kaggle/working/ver_topic_labeler_merged")